4.2 Extension 1 (Methodological): Semantic Supervision (2.5 Points)
This extension addresses the methodological critique raised in section 2.2: the weakness of Fuzzy Matching.
• The Concept: Instead of training SigExt on labels generated via simple character overlap, we will generate labels based on semantic similarity.
• Implementation:
• We will use a Sentence-BERT model (e.g., all-MiniLM-L6-v2 or paraphrase-multilingual-mpnet-base-v2) to encode all sentences of the source document (P_src) and all sentences of the target summary (P_ref) into dense vectors (embeddings).
• We will calculate the Cosine Similarity between the vectors.
• We will define a new labeling function:
• $$\\text{Label}(p) = 1 \\iff \\max\_{q \\in P\_{ref}} \\text{CosSim}(\\vec{p}, \\vec{q}) > \\theta$$
• where
theta is an experimental threshold (e.g., 0.6).
• Scientific Value: This modification transforms the extraction paradigm from "lexical" to "semantic." The hypothesis is that SigExt will learn to extract concepts, not just words, improving the quality of the signal sent to the LLM. This is an original scientific contribution that elevates the project above simple reproduction.


In [ ]:
#this is the dataset we are going to use for the first test: https://huggingface.co/datasets/ccdv/arxiv-summarization
!pip install datasets sentence-transformers nltk scikit-learn

from datasets import load_dataset

dataset = load_dataset(
    "ccdv/arxiv-summarization",
    "section",
    split="train"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00015.parquet:   0%|          | 0.00/230M [00:00<?, ?B/s]

section/train-00001-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00002-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

...

section/validation-00000-of-00001.parque(…):   0%|          | 0.00/105M [00:00<?, ?B/s]

section/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
from sklearn.metrics.pairwise import cosine_similarity
CONFIG = {
    'BERT_MODEL': 'all-MiniLM-L6-v2',
    'THRESHOLD': 0.6
}

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#articles = dataset["article"]
#abstract = dataset["abstract"]


In [ ]:


#article_sentences = []
#for article_text in articles:
#    article_sentences.extend(sent_tokenize(article_text))

#abstract_sentences = []
#for abstract_text in abstract:
#    abstract_sentences.extend(sent_tokenize(abstract_text))


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
#embs_article = embedder.encode(article_sentences)
#embs_abstract = embedder.encode(abstract_sentences)


In [ ]:
#import numpy as np

#embs_articles = np.array(embs_article)
#embs_abstracts = np.array(embs_abstract)

#def normalize(v):
#  return v / np.linalg.norm(v, axis=1, keepdims=True)

#embs_article = normalize(embs_article)
#embs_abstracts = normalize(embs_abstracts)

#cosine_sim = embs_articles @ embs_abstracts.T
#print(cosine_sim)

In [ ]:
example = dataset[0]

article_text = example["article"]
abstract_text = example["abstract"]


In [ ]:
import re

def is_good_sentence(s: str) -> bool:
    s = s.strip()

    if len(s) < 20:
        return False

    if re.fullmatch(r"[\W_]+", s):
        return False

    letters = sum(ch.isalpha() for ch in s)
    if letters < 10:
        return False
    if letters / len(s) < 0.25:
        return False

    if s in {"?", "??", "...", "?...", "?.", "??."}:
        return False

    return True

NameError: name 'article_text' is not defined

In [ ]:
# pip install datasets sentence-transformers nltk scikit-learn
import nltk
nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import nltk
import numpy as np

nltk.download("punkt")
from nltk.tokenize import sent_tokenize

CONFIG = {
    "DATASET_NAME": "ccdv/arxiv-summarization",
    "DATASET_CONFIG": "section",
    "SPLIT": "train",
    "BERT_MODEL": "sentence-transformers/all-MiniLM-L6-v2",
    "THRESHOLD": 0.6,
    "MAX_ARTICLE_SENTS": 200,
    "MAX_ABSTRACT_SENTS": 50,
}

dataset = load_dataset("ccdv/arxiv-summarization", "section", split="train")
embedder = SentenceTransformer(CONFIG["BERT_MODEL"])

def l2_normalize(x: np.ndarray) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

def semantic_labels(article_text: str, abstract_text: str, theta: float = 0.6):
    P_src = sent_tokenize(article_text)
    P_ref = sent_tokenize(abstract_text)

    P_src = [s for s in P_src if is_good_sentence(s)]
    P_ref = [s for s in P_ref if is_good_sentence(s)]

    P_src = P_src[:CONFIG["MAX_ARTICLE_SENTS"]]
    P_ref = P_ref[:CONFIG["MAX_ABSTRACT_SENTS"]]

    if len(P_src) == 0 or len(P_ref) == 0:
        return P_src, np.zeros((len(P_src),), dtype=int), np.zeros((len(P_src),), dtype=float)

    E_src = embedder.encode(P_src, batch_size=64, convert_to_numpy=True, show_progress_bar=False)
    E_ref = embedder.encode(P_ref, batch_size=64, convert_to_numpy=True, show_progress_bar=False)

    E_src = l2_normalize(E_src)
    E_ref = l2_normalize(E_ref)

    sim = E_src @ E_ref.T                      
    max_sim = sim.max(axis=1)                 

    y = (max_sim > theta).astype(int)         

    return P_src, y, max_sim

for i in range(500):
    P_src, y, max_sim = semantic_labels(dataset[i]["article"], dataset[i]["abstract"], CONFIG["THRESHOLD"])
    print(f"\nExample {i}: article_sents={len(P_src)}, positives={y.sum()}")
    top_idx = np.argsort(-max_sim)[:5]
    for k in top_idx:
        print(f"  score={max_sim[k]:.3f}  label={y[k]}  sent={P_src[k][:120]}...")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Example 0: article_sents=147, positives=20
  score=0.893  label=1  sent=in the last years 
 many interesting results on learning rates of regularized kernel based models for additive models ha...
  score=0.807  label=1  sent=in this paper we provide some learning rates for the support vector machines generated by additive kernels for additive ...
  score=0.759  label=1  sent=section [ ratessection ] contains our main results on learning rates for svms based on additive kernels ....
  score=0.751  label=1  sent=additive models @xcite provide an important family of models for semiparametric regression or classification ....
  score=0.748  label=1  sent=our learning rate in theorem [ quantilethm ] is new and optimal in the literature of svm for quantile regression ....

Example 1: article_sents=124, positives=20
  score=0.802  label=1  sent=in summary , using the sample of @xmath152 tagged @xmath28 decays with the cleo - c detector we obtain the absolute bran...
  score=0.774  label=1  s

In [ ]:
def make_semantic_supervision(article_text: str, abstract_text: str, theta: float):

    P_src = [s for s in sent_tokenize(article_text) if is_good_sentence(s)]
    P_ref = [s for s in sent_tokenize(abstract_text) if is_good_sentence(s)]
    P_src = P_src[:CONFIG["MAX_ARTICLE_SENTS"]]
    P_ref = P_ref[:CONFIG["MAX_ABSTRACT_SENTS"]]

    if len(P_src) == 0 or len(P_ref) == 0:
        return P_src, [], [], 0

    E_src = embedder.encode(P_src, batch_size=64, convert_to_numpy=True, show_progress_bar=False)
    E_ref = embedder.encode(P_ref, batch_size=64, convert_to_numpy=True, show_progress_bar=False)

    E_src = l2_normalize(E_src)
    E_ref = l2_normalize(E_ref)

    sim = E_src @ E_ref.T
    max_sim = sim.max(axis=1)
    labels = (max_sim > theta).astype(int)

    return P_src, labels.tolist(), max_sim.astype(float).tolist(), int(labels.sum())

def build_training_dataset(n_examples: int = 5000, save_path: str = "arxiv_semantic_supervision"):
    ds = load_dataset(CONFIG["DATASET_NAME"], CONFIG["DATASET_CONFIG"], split=CONFIG["SPLIT"])

    if n_examples is not None:
        ds = ds.select(range(min(n_examples, len(ds))))

    out = {
        "paper_id": [],
        "sentences": [],
        "labels": [],
        "scores": [],
        "num_sentences": [],
        "num_positives": [],
        "positive_rate": [],
        "abstract" : [],
    }

    for i, ex in enumerate(ds):
        paper_id = ex.get("paper_id", str(i))
        sents, labels, scores, n_pos = make_semantic_supervision(
            ex["article"], ex["abstract"], CONFIG["THRESHOLD"]
        )

        n_sents = len(sents)
        pos_rate = float(n_pos / n_sents) if n_sents > 0 else 0.0

        out["paper_id"].append(paper_id)
        out["sentences"].append(sents)
        out["labels"].append(labels)
        out["scores"].append(scores)
        out["num_sentences"].append(n_sents)
        out["num_positives"].append(n_pos)
        out["positive_rate"].append(pos_rate)
        out["abstract"].append(ex["abstract"])

        if (i + 1) % 200 == 0:
            print(f"Processed {i+1}/{len(ds)} examples...")

    train_ds = dataset.from_dict(out)

    train_ds.save_to_disk(save_path)
    print(f"Saved training dataset to: {save_path}")
    print(train_ds)
    return train_ds

train_ds = build_training_dataset(n_examples=5000, save_path="arxiv_semantic_supervision_theta06")

Processed 200/5000 examples...
Processed 400/5000 examples...
Processed 600/5000 examples...
Processed 800/5000 examples...
Processed 1000/5000 examples...
Processed 1200/5000 examples...
Processed 1400/5000 examples...
Processed 1600/5000 examples...
Processed 1800/5000 examples...
Processed 2000/5000 examples...
Processed 2200/5000 examples...
Processed 2400/5000 examples...
Processed 2600/5000 examples...
Processed 2800/5000 examples...
Processed 3000/5000 examples...
Processed 3200/5000 examples...
Processed 3400/5000 examples...
Processed 3600/5000 examples...
Processed 3800/5000 examples...
Processed 4000/5000 examples...
Processed 4200/5000 examples...
Processed 4400/5000 examples...
Processed 4600/5000 examples...
Processed 4800/5000 examples...
Processed 5000/5000 examples...


Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Saved training dataset to: arxiv_semantic_supervision_theta06
Dataset({
    features: ['paper_id', 'sentences', 'labels', 'scores', 'num_sentences', 'num_positives', 'positive_rate', 'abstract'],
    num_rows: 5000
})


{'paper_id': '0',
 'sentences': ['additive models @xcite provide an important family of models for semiparametric regression or classification .',
  'some reasons for the success of additive models are their increased flexibility when compared to linear or generalized linear models and their increased interpretability when compared to fully nonparametric models .',
  'it is well - known that good estimators in additive models are in general less prone to the curse of high dimensionality than good estimators in fully nonparametric models .',
  'many examples of such estimators belong to the large class of regularized kernel based methods over a reproducing kernel hilbert space @xmath0 , see e.g.',
  'in the last years \n many interesting results on learning rates of regularized kernel based models for additive models have been published when the focus is on sparsity and when the classical least squares loss function is used , see e.g.',
  '@xcite , @xcite , @xcite , @xcite , @xcite , @x

In [ ]:
pip install -q torch datasets sentence-transformers rouge-score

  Preparing metadata (setup.py) ... done


In [ ]:
import math
import random
from dataclasses import dataclass
from typing import List, Dict, Any

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset as TorchDataset, DataLoader

from datasets import load_from_disk
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer


CFG = {
    "DATA_PATH": "arxiv_semantic_supervision_theta06",  
    "EMBED_MODEL": "sentence-transformers/all-MiniLM-L6-v2",
    "BATCH_SIZE": 256,
    "LR": 2e-4,
    "EPOCHS": 3,
    "SEED": 42,
    "TRAIN_SPLIT": 0.9,       
    "TOP_K": 8,                
    "MAX_SUMMARY_WORDS": 200,  
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
}

random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])

ds = load_from_disk(CFG["DATA_PATH"])

if "abstract" not in ds.column_names:
    raise ValueError(
        "Your saved dataset does NOT include 'abstract'. "
        "ROUGE needs reference summaries. Rebuild the dataset including 'abstract' column."
    )

def flatten_for_training(hfds):
    paper_ids = []
    sentences = []
    labels = []
    paper_index = []  

    for pi, ex in enumerate(hfds):
        sents = ex["sentences"]
        labs = ex["labels"]
        if len(sents) != len(labs):
            continue
        for s, y in zip(sents, labs):
            paper_ids.append(ex.get("paper_id", str(pi)))
            paper_index.append(pi)
            sentences.append(s)
            labels.append(int(y))

    return {
        "paper_id": paper_ids,
        "paper_index": paper_index,
        "sentence": sentences,
        "label": labels,
    }

flat = flatten_for_training(ds)
N = len(flat["sentence"])
idx = np.arange(N)
np.random.shuffle(idx)

train_cut = int(CFG["TRAIN_SPLIT"] * N)
train_idx = idx[:train_cut]
val_idx = idx[train_cut:]


class SentenceLabelDataset(TorchDataset):
    def __init__(self, flat_dict, indices, embedder: SentenceTransformer):
        self.flat = flat_dict
        self.indices = indices
        self.embedder = embedder

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = int(self.indices[i])
        sent = self.flat["sentence"][j]
        y = float(self.flat["label"][j])
        return sent, y

@dataclass
class CollateEmbeddings:
    embedder: SentenceTransformer
    device: str

    def __call__(self, batch):
        sents = [b[0] for b in batch]
        ys = torch.tensor([b[1] for b in batch], dtype=torch.float32)
        embs = self.embedder.encode(
            sents, batch_size=len(sents), convert_to_numpy=True, show_progress_bar=False
        )
        embs = torch.tensor(embs, dtype=torch.float32)
        return embs.to(self.device), ys.to(self.device)


class SigExtHead(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(dim, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

embedder = SentenceTransformer(CFG["EMBED_MODEL"])
embed_dim = embedder.get_sentence_embedding_dimension()

model = SigExtHead(embed_dim).to(CFG["DEVICE"])
opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"])
crit = nn.BCEWithLogitsLoss()

train_ds = SentenceLabelDataset(flat, train_idx, embedder)
val_ds = SentenceLabelDataset(flat, val_idx, embedder)

collate = CollateEmbeddings(embedder=embedder, device=CFG["DEVICE"])

train_loader = DataLoader(train_ds, batch_size=CFG["BATCH_SIZE"], shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=CFG["BATCH_SIZE"], shuffle=False, collate_fn=collate)

def eval_sentence_level():
    model.eval()
    losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb)
            loss = crit(logits, yb)
            losses.append(loss.item())
    return float(np.mean(losses)) if losses else float("nan")

for epoch in range(1, CFG["EPOCHS"] + 1):
    model.train()
    running = []
    for xb, yb in train_loader:
        opt.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        running.append(loss.item())

    val_loss = eval_sentence_level()
    print(f"Epoch {epoch}/{CFG['EPOCHS']}  train_loss={np.mean(running):.4f}  val_loss={val_loss:.4f}")


torch.save(model.state_dict(), "sigext_head.pt")
print("Saved sigext_head.pt")


scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def make_extractive_summary(sentences: List[str], scores: List[float], top_k: int, max_words: int):
    ranked = sorted(list(enumerate(scores)), key=lambda x: x[1], reverse=True)[:top_k]
    keep_idx = sorted([i for i, _ in ranked])
    chosen = [sentences[i] for i in keep_idx]

    out = []
    wc = 0
    for s in chosen:
        w = s.split()
        if wc + len(w) > max_words:
            break
        out.append(s)
        wc += len(w)
    return " ".join(out).strip()

def predict_scores_for_sentences(sentences: List[str]) -> List[float]:
    if len(sentences) == 0:
        return []
    embs = embedder.encode(sentences, batch_size=64, convert_to_numpy=True, show_progress_bar=False)
    xb = torch.tensor(embs, dtype=torch.float32).to(CFG["DEVICE"])
    with torch.no_grad():
        logits = model(xb).detach().cpu().numpy()
        probs = 1.0 / (1.0 + np.exp(-logits))
    return probs.tolist()

def rouge_metrics(pred: str, ref: str) -> Dict[str, float]:
    scores = scorer.score(ref, pred) 
    r1 = scores["rouge1"]
    rl = scores["rougeL"]
    return {
        "R1-f": r1.fmeasure,
        "RL-f": rl.fmeasure,
        "R1-r": r1.recall,
    }

EVAL_PAPERS = min(500, len(ds))
metrics = {"R1-f": [], "RL-f": [], "R1-r": []}

model.eval()
for i in range(EVAL_PAPERS):
    ex = ds[i]
    sents = ex["sentences"]
    ref = ex["abstract"]

    probs = predict_scores_for_sentences(sents)

    pred = make_extractive_summary(
        sentences=sents,
        scores=probs,
        top_k=CFG["TOP_K"],
        max_words=CFG["MAX_SUMMARY_WORDS"],
    )

    m = rouge_metrics(pred, ref)
    for k in metrics:
        metrics[k].append(m[k])

print("\nROUGE results (mean over papers):")
for k, v in metrics.items():
    print(f"{k}: {np.mean(v):.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/3  train_loss=0.3733  val_loss=0.3606
Epoch 2/3  train_loss=0.3518  val_loss=0.3521
Epoch 3/3  train_loss=0.3413  val_loss=0.3475
Saved sigext_head.pt

ROUGE results (mean over papers):
R1-f: 0.4045
RL-f: 0.2125
R1-r: 0.4593
